In [0]:
%sql
CREATE TABLE workspace.streaming.sourcetable
(

  color STRING

)
using delta

In [0]:
from delta.tables import DeltaTable

In [0]:
DeltaTable.createIfNotExists(spark).\
  tableName("workspace.streaming.sourcetable"). \
  addColumn("color", "STRING")\
 .execute()

In [0]:
DeltaTable.createIfNotExists(spark).\
  tableName("workspace.streaming.sinktable"). \
  addColumn("color", "STRING")\
 .execute()

In [0]:
%sql
insert into workspace.streaming.sourcetable values 
('red'),
('green'),
('blue'),
('yellow'),
('orange'),
('orange')

num_affected_rows,num_inserted_rows
3,3


In [0]:
df=spark.readStream.table("workspace.streaming.sourcetable")


In [0]:
from pyspark.sql.functions import *

In [0]:
df=df.groupBy("color").agg(count("*").alias("count"))

In [0]:
df.writeStream.format("delta")\
    .outputMode("complete")\
    .trigger(once=True)\
    .option("checkpointLocation", "/Volumes/workspace/streaming/stream_volume/puntoControl/")\
    .option("path", "/Volumes/workspace/streaming/stream_volume/output/data")\
    .start()

In [0]:
%sql
SELECT * from delta.`/Volumes/workspace/streaming/stream_volume/output/data/`

color,count
yellow,1
orange,2
green,2
blue,2
red,2


In [0]:
%sql
insert into workspace.streaming.sourcetable values 
('magenta')


num_affected_rows,num_inserted_rows
1,1


In [0]:
df.writeStream.format("delta")\
    .outputMode("update")\
    .trigger(once=True)\
    .option("checkpointLocation", "/Volumes/workspace/streaming/stream_volume/puntoControl/")\
    .option("path", "/Volumes/workspace/streaming/stream_volume/output/data")\
    .start()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5861982394707235>, line 6
      1 df.writeStream.format("delta")\
      2     .outputMode("append")\
      3     .trigger(once=True)\
      4     .option("checkpointLocation", "/Volumes/workspace/streaming/stream_volume/puntoControl/")\
      5     .option("path", "/Volumes/workspace/streaming/stream_volume/output/data")\
----> 6     .start()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/streaming/readwriter.py:668, in DataStreamWriter.start(self, path, format, outputMode, partitionBy, queryName, **options)
    659 def start(
    660     self,
    661     path: Optional[str] = None,
   (...)
    666     **options: "OptionalPrimitiveType",
    667 ) -> StreamingQuery:
--> 668     return self._start_internal(
    669         path=path,
    670         tableName=None,
    671         format=format,